# Point picking

Every `k3d.points` cloud can report the point under the cursor. Two things have to be in
place: the plot has to be in `callback` mode, and the object needs a `click_callback` or a
`hover_callback`. Attaching either one builds a BVH over the point positions, so picking
stays fast on clouds far larger than this one.

The payload carries the index of the point, which is what makes the cloud able to react to
its own picks - here the point under the cursor turns yellow.

In [ ]:
import ipywidgets as widgets
import numpy as np

import k3d

N = 3000
t = np.linspace(0, 8 * np.pi, N, dtype=np.float32)
r = 1.0 + 0.3 * np.cos(7 * t)

positions = np.stack(
    [r * np.cos(t), r * np.sin(t), np.linspace(-1.5, 1.5, N)], axis=1
).astype(np.float32)

ramp = np.linspace(0, 1, N)
base_colors = (
    ((255 * ramp).astype(np.uint32) << 16)
    + (0x40 << 8)
    + (255 * (1 - ramp)).astype(np.uint32)
).astype(np.uint32)

plot = k3d.plot()
points = k3d.points(positions, colors=base_colors, point_size=0.06, shader='3d')
plot += points
plot.display()

info = widgets.HTML()
display(info)

In [ ]:
plot.mode = 'callback'

HIGHLIGHT = 0xffff00
highlighted = None


def on_pick(params):
    global highlighted

    index = params['index']

    if index == highlighted:
        return

    colors = points.colors.copy()

    if highlighted is not None:
        colors[highlighted] = base_colors[highlighted]

    colors[index] = HIGHLIGHT
    points.colors = colors
    highlighted = index

    info.value = '<pre>index:    %d\nposition: %s\ndistance: %.3f</pre>' % (
        index,
        np.round(params['position'], 3).tolist(),
        params['distance'],
    )


points.hover_callback = on_pick
points.click_callback = on_pick

Hover over the cloud - the point under the cursor turns yellow and its index, world
position and distance from the camera show up under the plot.

Both point shaders are pickable. `3d` draws camera-facing impostors and `mesh` draws real
spheres, but the callback payload is the same, and the pick radius follows `point_size` in
either case.

In [ ]:
points.shader = 'mesh'

Clearing both callbacks disarms picking: the cloud drops out of raycasting and releases
its BVH.

In [ ]:
points.click_callback = None
points.hover_callback = None